# RNA-PhysicsNet — Streaming Training on Colab T4

This notebook uses **hybrid on-demand data streaming** to train RNA-PhysicsNet on
Colab free tier with limited disk (~78 GB) and RAM (~12 GB).

- CIF and MSA files are downloaded from Kaggle **one chunk at a time** and deleted after use.
- CSV labels are read in streaming chunks (never fully loaded into RAM).
- A configurable disk-cache limit keeps total on-disk usage under ~10 GB.
- Training auto-resumes from the latest checkpoint on Colab disconnects.

### Phases
1. **Phase 1** — Pre-train on PDB RNA CIF structures (streamed from Kaggle)
2. **Phase 2** — Fine-tune on competition CSV data (lazy label loading)
3. **Phase 3** — Add MSA features (MSA files streamed on demand)
4. Export ONNX, validate, save to Drive

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch onnxruntime numba structlog gemmi tqdm psutil kaggle

In [ ]:
# Cell 2: Configure Kaggle credentials
#
# Option A — Colab Secrets (recommended):
#   Add KAGGLE_USERNAME and KAGGLE_KEY as Colab secrets (left sidebar → key icon).
#
# Option B — Upload kaggle.json:
#   from google.colab import files
#   files.upload()  # upload kaggle.json
#   !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

import os

try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print('Kaggle credentials loaded from Colab secrets.')
except Exception:
    print('Could not load Kaggle credentials from secrets — set KAGGLE_USERNAME / KAGGLE_KEY manually.')

# Mount Google Drive for saving checkpoints / ONNX
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/rna_physicsnet'
    os.makedirs(DRIVE_DIR, exist_ok=True)
except ImportError:
    DRIVE_DIR = '/tmp/rna_physicsnet'
    print('Not running on Colab — saving to /tmp/rna_physicsnet')

# Add repo to path
import sys
sys.path.insert(0, '.')

In [ ]:
# Cell 3: Download ONLY the small CSV files from Kaggle (not the 310 GB of CIFs/MSA)
KAGGLE_DATASET = 'yuion55/stanford-rna-3d-folding'
DATA_DIR = '/content/rna_data'
os.makedirs(DATA_DIR, exist_ok=True)

CSV_FILES = [
    'train_sequences.csv',
    'train_labels.csv',
    'validation_sequences.csv',
    'validation_labels.csv',
    'test_sequences.csv',
    'sample_submission.csv',
]

for csv_file in CSV_FILES:
    dest = os.path.join(DATA_DIR, csv_file)
    if not os.path.isfile(dest):
        print(f'Downloading {csv_file}...')
        ret = os.system(
            f'kaggle datasets download -d {KAGGLE_DATASET} -f {csv_file} -p {DATA_DIR} --unzip'
        )
        if ret != 0:
            print(f'  WARNING: could not download {csv_file}')
    else:
        print(f'  {csv_file} already present.')

TRAIN_CSV = os.path.join(DATA_DIR, 'train_sequences.csv')
LABELS_CSV = os.path.join(DATA_DIR, 'train_labels.csv')
VAL_CSV    = os.path.join(DATA_DIR, 'validation_sequences.csv')
VAL_LABELS = os.path.join(DATA_DIR, 'validation_labels.csv')
TEST_CSV   = os.path.join(DATA_DIR, 'test_sequences.csv')

print('\nCSV files available:')
for f in CSV_FILES:
    p = os.path.join(DATA_DIR, f)
    exists = os.path.isfile(p)
    size_kb = os.path.getsize(p) / 1024 if exists else 0
    print(f'  {f}: {"OK" if exists else "MISSING"} ({size_kb:.0f} KB)')

In [ ]:
# Cell 4: Import and configure with streaming enabled
import gc
import shutil

from topomatrix_rna.config import PipelineConfig, MemoryConfig
from topomatrix_rna.train import Trainer

config = PipelineConfig()

# Streaming configuration — keeps disk usage ≤ 8 GB at all times
config.memory = MemoryConfig(
    vram_gb=15.0,
    enable_streaming=True,
    max_disk_cache_gb=8.0,
    streaming_chunk_size=20,
    kaggle_dataset=KAGGLE_DATASET,
    data_source='kaggle',
    ram_limit_gb=10.0,
)

trainer = Trainer(config)
print(f'Model parameters: {sum(p.numel() for p in trainer.model.parameters()):,}')
print(f'Device: {trainer.device}')
print(f'Streaming: {config.memory.enable_streaming}')
print(f'Max disk cache: {config.memory.max_disk_cache_gb} GB')

# Helper: monitor RAM and disk
def print_resource_usage():
    try:
        import psutil
        ram_gb = psutil.virtual_memory().used / 1e9
        disk_gb = shutil.disk_usage('/').used / 1e9
        print(f'RAM used: {ram_gb:.1f} GB | Disk used: {disk_gb:.1f} GB')
    except ImportError:
        print('(psutil not available for resource monitoring)')

print_resource_usage()

In [ ]:
# Cell 5: Phase 1 — Pre-train on CIF structures (streaming)
#
# We pass a list of CIF filenames (relative paths within the Kaggle dataset)
# to phase1_pretrain.  They are downloaded in chunks of 20, parsed, then deleted.
#
# To get the full 9500+ file list without downloading them:
#   !kaggle datasets files -d {KAGGLE_DATASET} | grep PDB_RNA > /tmp/cif_list.txt
# or build the list from train_sequences.csv if pdb_ids are recorded there.
#
# For a quick test, we use only a small list below.

import csv

RESUME_PHASE1 = os.path.join(DRIVE_DIR, 'phase1_latest.pt')
CIF_LIST_FILE = '/tmp/cif_list.txt'  # one CIF filename per line

# Build CIF filename list (try from file, otherwise empty — will skip streaming)
cif_filenames = []
if os.path.isfile(CIF_LIST_FILE):
    with open(CIF_LIST_FILE) as f:
        cif_filenames = [line.strip() for line in f if line.strip().endswith('.cif')]
    print(f'Loaded {len(cif_filenames)} CIF filenames from {CIF_LIST_FILE}')
else:
    print(f'{CIF_LIST_FILE} not found — skipping Phase 1 streaming.')
    print('To populate it, run:')
    print(f'  !kaggle datasets files -d {KAGGLE_DATASET} | grep ".cif" | awk \'{{print $1}}\' > {CIF_LIST_FILE}')

if cif_filenames:
    print(f'Starting Phase 1 with {len(cif_filenames)} CIF files (chunk size {config.memory.streaming_chunk_size})...')
    print_resource_usage()
    trainer.phase1_pretrain(
        cif_dir='',           # unused in streaming mode
        n_epochs=50,
        cif_filenames=cif_filenames,
        resume_checkpoint=RESUME_PHASE1 if os.path.isfile(RESUME_PHASE1) else None,
    )
    # Save to Drive
    import shutil as _shutil
    trainer._save_checkpoint('phase1_latest.pt')
    _src = os.path.join(config.output_dir, 'phase1_latest.pt')
    if os.path.isfile(_src):
        _shutil.copy2(_src, RESUME_PHASE1)
    print('Phase 1 done.')
    print_resource_usage()
else:
    print('Phase 1 skipped (no CIF filename list).')

In [ ]:
# Cell 6: Phase 2 — Fine-tune on competition CSV data (lazy label loading)
RESUME_PHASE2 = os.path.join(DRIVE_DIR, 'phase2_latest.pt')

if os.path.isfile(TRAIN_CSV) and os.path.isfile(LABELS_CSV):
    print('Starting Phase 2: Competition data fine-tuning (lazy label streaming)...')
    print_resource_usage()
    trainer.phase2_finetune(
        TRAIN_CSV, LABELS_CSV,
        val_csv=VAL_CSV if os.path.isfile(VAL_CSV) else None,
        val_labels=VAL_LABELS if os.path.isfile(VAL_LABELS) else None,
        n_epochs=100,
        resume_checkpoint=RESUME_PHASE2 if os.path.isfile(RESUME_PHASE2) else None,
    )
    import shutil as _shutil
    trainer._save_checkpoint('phase2_latest.pt')
    _src = os.path.join(config.output_dir, 'phase2_latest.pt')
    if os.path.isfile(_src):
        _shutil.copy2(_src, RESUME_PHASE2)
    print('Phase 2 done.')
    print_resource_usage()
else:
    print(f'Training CSV not found at {TRAIN_CSV}')
    print('Skipping Phase 2')

In [ ]:
# Cell 7: Phase 3 — MSA-augmented training (streaming MSA downloads)
RESUME_PHASE3 = os.path.join(DRIVE_DIR, 'phase3_latest.pt')
MSA_DIR = '/tmp/msa_placeholder'  # unused in streaming mode; files come from Kaggle

if os.path.isfile(TRAIN_CSV) and os.path.isfile(LABELS_CSV):
    print('Starting Phase 3: MSA-augmented training (streaming MSA downloads)...')
    print_resource_usage()
    trainer.phase3_msa(
        TRAIN_CSV, LABELS_CSV,
        msa_dir=MSA_DIR,
        n_epochs=50,
        resume_checkpoint=RESUME_PHASE3 if os.path.isfile(RESUME_PHASE3) else None,
    )
    import shutil as _shutil
    trainer._save_checkpoint('phase3_latest.pt')
    _src = os.path.join(config.output_dir, 'phase3_latest.pt')
    if os.path.isfile(_src):
        _shutil.copy2(_src, RESUME_PHASE3)
    print('Phase 3 done.')
    print_resource_usage()
else:
    print(f'Training CSV not found — skipping Phase 3')

In [ ]:
# Cell 8: Export ONNX
ONNX_OUT = 'topomatrix_rna/models/rna_physicsnet.onnx'
onnx_path = trainer.export_onnx(ONNX_OUT)
print(f'ONNX model exported to: {onnx_path}')
print(f'File size: {os.path.getsize(onnx_path) / 1024 / 1024:.1f} MB')
print_resource_usage()

In [ ]:
# Cell 9: Validate on validation set
if os.path.isfile(VAL_CSV) and os.path.isfile(VAL_LABELS):
    from topomatrix_rna.memory_manager import CompetitionDataset
    import torch
    from torch.utils.data import DataLoader

    val_dataset = CompetitionDataset(VAL_CSV, VAL_LABELS)
    val_loader = DataLoader(val_dataset, batch_size=1, num_workers=0)

    mean_tm = trainer.validate(val_loader)
    print(f'Validation TM-score: {mean_tm:.4f}')

    del val_dataset, val_loader
    gc.collect()
else:
    print('Validation data not available')
print_resource_usage()

In [ ]:
# Cell 10: Save ONNX and final checkpoint to Drive
import shutil

if os.path.isfile(onnx_path):
    drive_onnx = os.path.join(DRIVE_DIR, 'rna_physicsnet.onnx')
    try:
        shutil.copy2(onnx_path, drive_onnx)
        print(f'ONNX copied to: {drive_onnx}')
    except Exception as e:
        print(f'Could not copy to Drive: {e}')
        print(f'ONNX is at: {onnx_path}')
else:
    print('No ONNX file found')

# Also copy best model checkpoint if it exists
best_ckpt = os.path.join(config.output_dir, 'best_model.pt')
if os.path.isfile(best_ckpt):
    shutil.copy2(best_ckpt, os.path.join(DRIVE_DIR, 'best_model.pt'))
    print(f'Best checkpoint saved to Drive.')

print_resource_usage()
print('All done!')